In [1]:
''' IMPORTS '''

import math
from datetime import datetime
import itertools

import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

import nfl_data_py as nfl

from resources.plotly_theme import nfl_template
from resources.get_nfl_data import get_matchups, get_team_info
from resources.epa_model import PicksModel, EPAUtility

pio.templates['nfl_template'] = nfl_template

In [2]:
''' Team info '''

# Get team info
team_data = get_team_info()

In [3]:
''' Get all possible matchups '''

playoff_teams = ['DEN', 'NE', 'JAX', 'PIT', 'HOU', 'BUF', 'LAC',
                 'SEA', 'CHI', 'PHI', 'CAR', 'LA', 'SF', 'GB']

all_possible_matchups = list(itertools.combinations(playoff_teams, 2))
print(all_possible_matchups)

df = pd.DataFrame.from_records(columns=['home_team', 'away_team'], data=all_possible_matchups)
print(df.shape)
print(df.head())
print(df.tail())

[('DEN', 'NE'), ('DEN', 'JAX'), ('DEN', 'PIT'), ('DEN', 'HOU'), ('DEN', 'BUF'), ('DEN', 'LAC'), ('DEN', 'SEA'), ('DEN', 'CHI'), ('DEN', 'PHI'), ('DEN', 'CAR'), ('DEN', 'LA'), ('DEN', 'SF'), ('DEN', 'GB'), ('NE', 'JAX'), ('NE', 'PIT'), ('NE', 'HOU'), ('NE', 'BUF'), ('NE', 'LAC'), ('NE', 'SEA'), ('NE', 'CHI'), ('NE', 'PHI'), ('NE', 'CAR'), ('NE', 'LA'), ('NE', 'SF'), ('NE', 'GB'), ('JAX', 'PIT'), ('JAX', 'HOU'), ('JAX', 'BUF'), ('JAX', 'LAC'), ('JAX', 'SEA'), ('JAX', 'CHI'), ('JAX', 'PHI'), ('JAX', 'CAR'), ('JAX', 'LA'), ('JAX', 'SF'), ('JAX', 'GB'), ('PIT', 'HOU'), ('PIT', 'BUF'), ('PIT', 'LAC'), ('PIT', 'SEA'), ('PIT', 'CHI'), ('PIT', 'PHI'), ('PIT', 'CAR'), ('PIT', 'LA'), ('PIT', 'SF'), ('PIT', 'GB'), ('HOU', 'BUF'), ('HOU', 'LAC'), ('HOU', 'SEA'), ('HOU', 'CHI'), ('HOU', 'PHI'), ('HOU', 'CAR'), ('HOU', 'LA'), ('HOU', 'SF'), ('HOU', 'GB'), ('BUF', 'LAC'), ('BUF', 'SEA'), ('BUF', 'CHI'), ('BUF', 'PHI'), ('BUF', 'CAR'), ('BUF', 'LA'), ('BUF', 'SF'), ('BUF', 'GB'), ('LAC', 'SEA'), ('LAC'

In [3]:
df = pd.DataFrame(data={
    'home_team': ['PHI'],
    'away_team': ['SF']
})
print(df)

  home_team away_team
0       PHI        SF


In [4]:
''' Predict All Possible Matchups '''

# Predict
picks_model = PicksModel()
predictions_df = picks_model.predict_matchups(matchups=df, moneyline_value=False)

print(predictions_df.shape)
print(predictions_df.to_string())

Begin add epa (91, 2)
2024 done.
2025 done.
Downcasting floats.
Downcasting floats.


/Users/jmiller/Documents/Fun/nfl/notebooks/resources/get_nfl_data.py:224: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pbp_data['Non-Play Type'] = conditions
/Users/jmiller/Documents/Fun/nfl/notebooks/resources/get_nfl_data.py:227: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pbp_data['Play Counted'] = (pbp_data['penalty_team'] != pbp_data['posteam'])
/Users/jmiller/Documents/Fun/nfl/notebooks/resources/get_nfl_data.py:230: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert

Begin add epa inputs to matchup (91, 3)
[999999]
   master_week team  Last_4_EPA_O  Last_4_EPA_D  Last_4_EPA_ST  Last_8_EPA_O  Last_8_EPA_D  Last_8_EPA_ST  Last_12_EPA_O  Last_12_EPA_D  Last_12_EPA_ST  Last_16_EPA_O  Last_16_EPA_D  Last_16_EPA_ST  Last_4_EPA_O_Play  Last_4_EPA_D_Play  Last_4_EPA_ST_Play  Last_8_EPA_O_Play  Last_8_EPA_D_Play  Last_8_EPA_ST_Play  Last_12_EPA_O_Play  Last_12_EPA_D_Play  Last_12_EPA_ST_Play  Last_16_EPA_O_Play  Last_16_EPA_D_Play  Last_16_EPA_ST_Play
0       999999  DEN      5.780742     23.049118     -11.400272     10.114262     28.894239       0.841167      24.742382      41.328362       -9.612988      50.438797      40.671574       -2.506261           0.022319           0.093696           -0.107550           0.019339           0.056214            0.004025            0.031680            0.052051            -0.028695            0.048406            0.038442            -0.005607
1       999999  PHI     30.334763     17.164104      -2.293590     -4.377495   

In [7]:
predictions_df = predictions_df[['home_team', 'away_team', 'prob_home', 'prob_away']]
print(predictions_df.shape)

opp = predictions_df.rename(columns={
    'home_team': 'away_team',
    'away_team': 'home_team',
    'prob_home': 'prob_away',
    'prob_away': 'prob_home',
})

df = pd.concat([predictions_df, opp]).reset_index(drop=True)
df = df.rename(columns={
    'home_team': 'team',
    'away_team': 'opponent',
    'prob_home': 'prob',
    'prob_away': 'opponent_prob'
})
print(df.shape)
print(df.head().to_string())

df.to_excel('Playoff Probs.xlsx')

(91, 4)
(182, 4)
  team opponent      prob  opponent_prob
0  DEN       NE  0.382748       0.617252
1  DEN      JAX  0.460961       0.539039
2  DEN      PIT  0.603941       0.396059
3  DEN      HOU  0.504875       0.495125
4  DEN      BUF  0.462619       0.537381
